# ML1 – Hand Gesture Classification with MLflow (research branch)

This notebook mirrors the baseline ML1 notebook, but adds MLflow tracking for each trained model.

- All MLflow logic lives in `mlflow_utils.py`.
- We log dataset info, parameters, metrics, and the trained model as artifacts.
- Runs are created under the `hand_gesture_classification` experiment.


In [ ]:
# If running in Colab, install dependencies (uncomment if needed)
# !pip install numpy pandas scikit-learn matplotlib seaborn joblib mlflow mediapipe opencv-python

import pandas as pd
from pathlib import Path

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
import seaborn as sns

from mlflow_utils import run_experiment

PROJECT_ROOT = Path("..").resolve()
DATA_PATH = PROJECT_ROOT / "data" / "hand_landmarks.csv"

In [ ]:
def load_hand_landmarks(csv_path: str) -> pd.DataFrame:
    return pd.read_csv(csv_path)


def recenter_and_normalize(df: pd.DataFrame, label_col: str = "label"):
    LANDMARK_COUNT = 21

    wrist_x = df["x1"]
    wrist_y = df["y1"]

    for i in range(1, LANDMARK_COUNT + 1):
        df[f"x{i}"] = df[f"x{i}"] - wrist_x
        df[f"y{i}"] = df[f"y{i}"] - wrist_y

    middle_x = df["x13"]
    middle_y = df["y13"]
    scale = (middle_x ** 2 + middle_y ** 2) ** 0.5
    scale = scale.replace(0, 1e-6)

    for i in range(1, LANDMARK_COUNT + 1):
        df[f"x{i}"] = df[f"x{i}"] / scale
        df[f"y{i}"] = df[f"y{i}"] / scale

    feature_cols = []
    for i in range(1, LANDMARK_COUNT + 1):
        feature_cols.extend([f"x{i}", f"y{i}", f"z{i}"])

    X = df[feature_cols]
    y = df[label_col]
    return X, y

In [ ]:
df = load_hand_landmarks(str(DATA_PATH))
X, y = recenter_and_normalize(df)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

In [ ]:
models = {
    "RandomForest": RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42),
    "SVM-RBF": SVC(kernel="rbf", C=10, gamma="scale", probability=True, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=15),
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    metrics = {
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_macro": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
    }

    print(f"=== {name} ===")
    print(classification_report(y_test, y_pred, zero_division=0))

    # Log this run to MLflow using the shared utilities.
    run_experiment(
        experiment_name="hand_gesture_classification",
        run_name=f"{name}_notebook_run",
        dataset_path=str(DATA_PATH),
        dataset_n_rows=len(df),
        model_name=name,
        model=model,
        model_params=model.get_params(),
        metric_values={
            "accuracy": metrics["accuracy"],
            "precision_macro": metrics["precision_macro"],
            "recall_macro": metrics["recall_macro"],
            "f1_macro": metrics["f1_macro"],
        },
        register_as=None,
    )

    results.append(metrics)

results_df = pd.DataFrame(results)
results_df

In [ ]:
plt.figure(figsize=(8, 5))
metrics_to_plot = ["accuracy", "precision_macro", "recall_macro", "f1_macro"]
long_df = results_df.melt(id_vars="model", value_vars=metrics_to_plot, var_name="metric")
sns.barplot(data=long_df, x="model", y="value", hue="metric")
plt.ylim(0, 1.0)
plt.title("Model comparison (macro metrics)")
plt.show()

best_row = results_df.sort_values("f1_macro", ascending=False).iloc[0]
print("Best model:")
best_row